In [ ]:
%pip install -q "qualtran @ git+https://github.com/quantumlib/Qualtran.git@c58703d00b01c346d3e77d3913cf2d6e9e556be3"


In [ ]:
import os, sys, json, hashlib, shutil, zipfile, traceback
from pathlib import Path

DATASET = Path('/kaggle/input/datasets/evanpieser/qcompile-verifiable')
COMP = Path('/kaggle/input/competitions/google-verifiable-quantum-circuit-compilation')
PUBLIC_CANDIDATE = DATASET / 'public_candidate'
SAMPLE = COMP / 'sample_submission'
WORK = Path('/kaggle/working')
OUT = WORK / 'submission'

sys.path.insert(0, str(COMP / 'kickmix'))
sys.path.insert(0, str(DATASET / 'starter_kit'))
sys.path.insert(0, str(DATASET))

os.environ['QCOMPILE_GENERATIONS'] = os.environ.get('QCOMPILE_GENERATIONS', '80')
os.environ['QCOMPILE_POP_SIZE'] = os.environ.get('QCOMPILE_POP_SIZE', '60')
os.environ['QCOMPILE_ENABLE'] = '1'

print('Full portfolio optimization starting...')
print('qcompile gens:', os.environ['QCOMPILE_GENERATIONS'], 'pop:', os.environ['QCOMPILE_POP_SIZE'])


In [ ]:
# Lock verified dfthcsos (do not regenerate)
shutil.rmtree(OUT, ignore_errors=True)
OUT.mkdir(parents=True)
(OUT / 'dfthcsos').mkdir(parents=True)

src_dfthc = PUBLIC_CANDIDATE / 'dfthcsos'
for p in src_dfthc.iterdir():
    if p.is_file():
        shutil.copy2(p, OUT / 'dfthcsos' / p.name)

# Start from baseline for others, qcompile will optimize if possible
shutil.copy2(SAMPLE / 'dqi.kmb', OUT / 'dqi.kmb')
shutil.copy2(SAMPLE / 'invsqrt.kmb', OUT / 'invsqrt.kmb')

print('Copied verified dfthcsos + baseline dqi/invsqrt')


In [ ]:
# Try to optimize dqi and invsqrt using qcompile if supported
try:
    from qcompile import QuantumCircuitSolver
    import kickmix as km

    solver = QuantumCircuitSolver(
        generations=int(os.environ['QCOMPILE_GENERATIONS']),
        population_size=int(os.environ['QCOMPILE_POP_SIZE'])
    )

    def load_circuit(path):
        path = str(path)
        # Try common kickmix APIs
        attempts = []
        if hasattr(km.Circuit, 'read'):
            attempts.append(lambda p: km.Circuit.read(p))
        if hasattr(km.Circuit, 'load'):
            attempts.append(lambda p: km.Circuit.load(p))
        if hasattr(km.Circuit, 'from_file'):
            attempts.append(lambda p: km.Circuit.from_file(p))
        if hasattr(km, 'read_circuit'):
            attempts.append(lambda p: km.read_circuit(p))
        if hasattr(km, 'load_circuit'):
            attempts.append(lambda p: km.load_circuit(p))
        if hasattr(km, 'read_kmb'):
            attempts.append(lambda p: km.read_kmb(p))
        if hasattr(km, 'load') and not hasattr(km.Circuit, 'load'):
            attempts.append(lambda p: km.load(p))
        try:
            import kickmix.io as kmio
            for name in ['read', 'load', 'read_circuit', 'read_kmb']:
                if hasattr(kmio, name):
                    fn = getattr(kmio, name)
                    attempts.append(lambda p, fn=fn: fn(p))
        except Exception:
            pass
        for attempt in attempts:
            try:
                return attempt(path)
            except Exception:
                continue
        raise RuntimeError('Could not load circuit with known kickmix APIs')

    def save_circuit(circ, path):
        path = str(path)
        if hasattr(circ, 'save'):
            circ.save(path)
            return
        if hasattr(circ, 'write'):
            circ.write(path)
            return
        # fallback: try to write as text if supported
        try:
            with open(path, 'w') as f:
                f.write(str(circ))
            return
        except Exception as e:
            raise e

    for name in ['dqi.kmb', 'invsqrt.kmb']:
        path = OUT / name
        try:
            circ = load_circuit(path)
            opt = solver.solve(circ)
            if opt is not None:
                save_circuit(opt, path)
                print(f'Optimized {name}')
            else:
                print(f'No improvement for {name}, keeping baseline')
        except Exception as e:
            print(f'Skipping {name} optimization:', e)
            traceback.print_exc()
except Exception as e:
    print('qcompile not applied:', e)
    traceback.print_exc()


In [ ]:
from verifier.verify import verify

verify_full = None
try:
    res = verify('FeMoCo76o113e', OUT / 'dfthcsos')
    print('dfthcsos:', res['status'], res['score'])
    verify_full = res
except Exception as e:
    print('verify err:', e)
    traceback.print_exc()

# Basic sanity
assert (OUT/'dqi.kmb').exists() and (OUT/'invsqrt.kmb').exists()
assert (OUT/'dfthcsos'/'submission.npz').exists()
print('Sanity checks passed')


In [ ]:
def sha256(p):
    import hashlib
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()

files = sorted([p for p in OUT.rglob('*') if p.is_file()])
members = {str(p.relative_to(OUT)): sha256(p) for p in files}
expected = {'dqi.kmb','invsqrt.kmb','dfthcsos/submission.npz'} | {f'dfthcsos/angle_k_{i}.kmb' for i in range(15)}
assert set(members) == expected, (set(members)^expected)

zip_path = WORK/'submission.zip'
tmp = WORK/'submission.tmp.zip'
with zipfile.ZipFile(tmp,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in files: z.write(p, str(p.relative_to(OUT)))
with zipfile.ZipFile(tmp) as z: 
    assert z.testzip() is None
os.replace(tmp, zip_path)

report = {'dfthcsos_verified': verify_full, 'members': members, 'archive_sha256': sha256(zip_path)}
(WORK/'full_portfolio_report.json').write_text(json.dumps(report,indent=2,default=float))
print('Wrote', zip_path)
print('Archive SHA256:', report['archive_sha256'])
